# Data Preparation & Feature Engineering Notebook

**Mục tiêu:** Tạo bảng dữ liệu daily đã được clean/có kiểm tra, tạo feature có kiểm soát leakage, và lưu các output cần thiết trước khi train model dự đoán `Revenue` và `COGS`.

**Phạm vi:** Notebook này chỉ làm data preparation và feature engineering; chưa train final model.

**Nguyên tắc:**
- Target grain: 1 dòng = 1 ngày.
- Mỗi bảng khác phải aggregate về ngày trước khi join.
- Feature có nguy cơ leakage chỉ được dùng dưới dạng lag/rolling từ quá khứ.
- Không sinh feature hàng loạt nếu không có lý do nghiệp vụ hoặc tín hiệu kiểm chứng.


## 0. Setup & Project Context

**Mục tiêu:** Khởi tạo thư viện, đường dẫn input/output và cấu hình hiển thị; bước này giúp notebook chạy lặp lại được và các file đầu ra nằm đúng thư mục dự án.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

sns.set_theme(style='whitegrid')

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'report_3_6_2026' else Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / 'data'
REPORT_DIR = PROJECT_DIR / 'report_3_6_2026'
OUTPUT_DIR = REPORT_DIR / 'feature_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


## 1. Problem Definition & Target Setup

### 1.1. Target Variables

**Mục tiêu:** Xác định target cần dự đoán là `Revenue` và `COGS` từ `sales.csv`; bước này giúp model không bị lệch mục tiêu và giúp các feature sau đó được tạo đúng theo bài toán dự báo doanh thu/chi phí theo ngày.


In [ ]:
tables = {}
for path in sorted(DATA_DIR.glob('*.csv')):
    tables[path.stem] = pd.read_csv(path, low_memory=False)
    print(f'{path.stem:18s}', tables[path.stem].shape)

required = {'sales', 'sample_submission'}
missing_required = required - set(tables)
if missing_required:
    raise ValueError(f'Missing required tables: {missing_required}')

target_variables = pd.DataFrame([
    {'target_column': 'Revenue', 'source_table': 'sales', 'meaning': 'Daily revenue to forecast'},
    {'target_column': 'COGS', 'source_table': 'sales', 'meaning': 'Daily cost of goods sold to forecast'},
])
display(target_variables)

target_column_check = pd.DataFrame({
    'required_column': ['Date', 'Revenue', 'COGS'],
    'exists_in_sales': [col in tables['sales'].columns for col in ['Date', 'Revenue', 'COGS']],
})
display(target_column_check)

if not target_column_check['exists_in_sales'].all():
    raise ValueError('sales.csv thieu cot target bat buoc')


### 1.2. Prediction Grain

**Mục tiêu:** Chốt grain dự báo là `1 row = 1 Date` và tạo modeling base từ `sales`; bước này giúp tất cả bảng nguồn sau này phải aggregate về cùng grain trước khi join, tránh nhân dòng và tính sai target.


In [ ]:
sales = tables['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'], errors='coerce')
sales = sales.sort_values('Date').reset_index(drop=True)

if sales['Date'].isna().any():
    raise ValueError('sales.Date co gia tri khong parse duoc')
if sales['Date'].duplicated().any():
    raise ValueError('sales.Date bi duplicate, khong dung grain 1 ngay')

modeling = sales[['Date', 'Revenue', 'COGS']].copy()

prediction_grain = pd.DataFrame([{
    'modeling_unit': 'daily',
    'grain': '1 row = 1 Date',
    'n_rows': len(modeling),
    'date_min': modeling['Date'].min(),
    'date_max': modeling['Date'].max(),
}])
display(prediction_grain)
display(modeling.head())
display(modeling.tail())


## 2. Modeling Table Design

### 2.1. Modeling Grain Validation

**Mục tiêu:** Kiểm tra `Date` trong modeling base có unique, không missing và có date range rõ ràng; bước này giúp xác nhận bảng train có đúng 1 dòng cho 1 ngày trước khi thêm feature.


In [ ]:
grain_check = pd.DataFrame([{
    'table': 'modeling_base_sales',
    'grain_expected': '1 row = 1 Date',
    'n_rows': len(modeling),
    'n_unique_date': modeling['Date'].nunique(),
    'duplicate_date_rows': int(modeling['Date'].duplicated().sum()),
    'date_min': modeling['Date'].min(),
    'date_max': modeling['Date'].max(),
}])
display(grain_check)

if grain_check.loc[0, 'duplicate_date_rows'] != 0:
    raise ValueError('Modeling base bi duplicate Date')


### 2.2. Date Coverage Check

**Mục tiêu:** Đo coverage theo ngày của `sales`, `sample_submission` và các source quan trọng; bước này giúp biết bảng nào đủ phủ để tạo feature, bảng nào cần skip hoặc chỉ dùng rất thận trọng.


In [ ]:
coverage_specs = [
    ('sales', 'Date'),
    ('sample_submission', 'Date'),
    ('orders', 'order_date'),
    ('web_traffic', 'date'),
    ('inventory', 'snapshot_date'),
    ('shipments', 'ship_date'),
    ('returns', 'return_date'),
    ('reviews', 'review_date'),
]
modeling_dates = set(modeling['Date'].dt.normalize())
coverage_rows = []
for table, date_col in coverage_specs:
    if table not in tables or date_col not in tables[table].columns:
        continue
    parsed_dates = pd.to_datetime(tables[table][date_col], errors='coerce').dropna().dt.normalize()
    unique_dates = parsed_dates.drop_duplicates()
    coverage_rows.append({
        'table': table,
        'date_col': date_col,
        'n_unique_dates': len(unique_dates),
        'date_min': unique_dates.min() if len(unique_dates) else pd.NaT,
        'date_max': unique_dates.max() if len(unique_dates) else pd.NaT,
        'target_date_coverage_rate': pd.Series(list(modeling_dates)).isin(set(unique_dates)).mean() if len(modeling_dates) else 0,
    })

date_coverage = pd.DataFrame(coverage_rows).sort_values('target_date_coverage_rate', ascending=False)
display(date_coverage)


## 3. Data Preparation Before Feature Engineering

### 3.1. Date Parsing & Time Alignment

**Mục tiêu:** Parse các cột ngày về đúng kiểu datetime và đưa về daily date; bước này giúp aggregate theo ngày chính xác và tránh join sai do format ngày khác nhau.


In [ ]:
date_checks = []
date_columns = {
    'sales': ['Date'],
    'orders': ['order_date'],
    'web_traffic': ['date'],
    'inventory': ['snapshot_date'],
    'promotions': ['start_date', 'end_date'],
    'shipments': ['ship_date', 'delivery_date'],
    'returns': ['return_date'],
    'reviews': ['review_date'],
}
for table, cols in date_columns.items():
    if table not in tables:
        continue
    for col in cols:
        if col not in tables[table].columns:
            continue
        parsed = pd.to_datetime(tables[table][col], errors='coerce')
        date_checks.append({
            'table': table,
            'column': col,
            'parse_or_missing_issue_count': int(parsed.isna().sum()),
            'n_rows': len(tables[table]),
            'issue_rate': float(parsed.isna().mean()),
        })

date_parse_check = pd.DataFrame(date_checks).sort_values('issue_rate', ascending=False)
display(date_parse_check)


### 3.2. Key Validation Before Join

**Mục tiêu:** Kiểm tra duplicate key và quan hệ join giữa các bảng chính; bước này giúp phát hiện nguy cơ one-to-many/many-to-many làm nhân dòng, đặc biệt ở `order_items`, `products`, `orders`.


In [ ]:
key_checks = []
for table, key in [('products', 'product_id'), ('customers', 'customer_id'), ('orders', 'order_id'), ('promotions', 'promo_id')]:
    if table in tables and key in tables[table].columns:
        key_checks.append({
            'table': table,
            'key_column': key,
            'n_rows': len(tables[table]),
            'n_unique_key': tables[table][key].nunique(dropna=True),
            'duplicate_key_rows': int(tables[table].duplicated([key]).sum()),
            'join_risk_note': 'duplicate key can inflate joins' if tables[table].duplicated([key]).any() else 'ok',
        })

key_validation = pd.DataFrame(key_checks)
display(key_validation)


### 3.3. Missing Value Treatment Strategy

**Mục tiêu:** Đánh giá missing theo ý nghĩa nghiệp vụ trước khi fill/drop; bước này giúp tránh xóa mất tín hiệu quan trọng, vì missing có thể là không có event thay vì lỗi dữ liệu.


In [ ]:
missing_value_strategy = pd.DataFrame([
    {'field_or_case': 'promo_id missing', 'decision': 'create has_promo flag; do not fill promo_id', 'reason': 'missing usually means no promotion was used'},
    {'field_or_case': 'shipment missing with cancelled/created/paid', 'decision': 'treat as likely valid business state', 'reason': 'these orders may not have been shipped yet or were cancelled'},
    {'field_or_case': 'shipment missing with delivered/returned/shipped', 'decision': 'flag as data quality issue if used later', 'reason': 'status implies fulfillment should exist'},
    {'field_or_case': 'high-missing source tables', 'decision': 'skip feature source if date coverage is too low', 'reason': 'high missing feature creates unstable model inputs'},
])
display(missing_value_strategy)


### 3.4. Outlier Review Strategy

**Mục tiêu:** Kiểm tra outlier ở các cột tiền, quantity, traffic và tỉ lệ; bước này giúp phân biệt giá trị bất thường hợp lý về kinh doanh với lỗi dữ liệu trước khi đưa vào model.


In [ ]:
outlier_review_strategy = pd.DataFrame([
    {'metric_family': 'Revenue/COGS', 'review_rule': 'high value must move together with historical demand or COGS', 'default_action': 'keep if business-plausible; do not cap automatically'},
    {'metric_family': 'order_items quantity/discount', 'review_rule': 'high value should be checked with quantity, promo flag and product mix', 'default_action': 'use aggregate demand features, avoid target proxy revenue features'},
    {'metric_family': 'refund/return', 'review_rule': 'high refund should be reviewed with return_count/return_quantity', 'default_action': 'use lag/rolling from past only'},
])
display(outlier_review_strategy)


**Technical helper functions**

Cell kỹ thuật để tạo feature và ghi feature catalog. Đây không phải một task heading riêng.


In [ ]:
feature_records = []
skipped_feature_sources = []
section_feature_reviews = []

def add_feature_record(feature_name, source_table, feature_group, logic, uses_lag, leakage_risk, business_reason):
    feature_records.append({
        'feature_name': feature_name,
        'source_table': source_table,
        'feature_group': feature_group,
        'logic_tao_feature': logic,
        'co_dung_lag_khong': uses_lag,
        'leakage_risk': leakage_risk,
        'business_reason': business_reason,
        'status': 'created',
    })

def record_skipped_source(source_table, reason, coverage_rate=None, n_dates=None):
    skipped_feature_sources.append({
        'source_table': source_table,
        'reason': reason,
        'coverage_rate': coverage_rate,
        'n_dates': n_dates,
    })

def build_vietnam_holiday_dates(years):
    holiday_dates = set()
    tet_first_days = {
        2012: '2012-01-23', 2013: '2013-02-10', 2014: '2014-01-31',
        2015: '2015-02-19', 2016: '2016-02-08', 2017: '2017-01-28',
        2018: '2018-02-16', 2019: '2019-02-05', 2020: '2020-01-25',
        2021: '2021-02-12', 2022: '2022-02-01', 2023: '2023-01-22',
        2024: '2024-02-10', 2025: '2025-01-29', 2026: '2026-02-17',
    }
    hung_kings_days = {
        2012: '2012-03-31', 2013: '2013-04-19', 2014: '2014-04-09',
        2015: '2015-04-28', 2016: '2016-04-16', 2017: '2017-04-06',
        2018: '2018-04-25', 2019: '2019-04-14', 2020: '2020-04-02',
        2021: '2021-04-21', 2022: '2022-04-10', 2023: '2023-04-29',
        2024: '2024-04-18', 2025: '2025-04-07', 2026: '2026-04-26',
    }
    for year in years:
        for fixed_day in [f'{year}-01-01', f'{year}-04-30', f'{year}-05-01', f'{year}-09-02']:
            holiday_dates.add(pd.Timestamp(fixed_day))
        if year in tet_first_days:
            tet_day = pd.Timestamp(tet_first_days[year])
            for offset in range(-2, 5):
                holiday_dates.add(tet_day + pd.Timedelta(days=offset))
        if year in hung_kings_days:
            holiday_dates.add(pd.Timestamp(hung_kings_days[year]))
    return holiday_dates

def add_calendar_features(df):
    out = df[['Date']].copy()
    out['dayofweek'] = out['Date'].dt.dayofweek
    out['month'] = out['Date'].dt.month
    out['quarter'] = out['Date'].dt.quarter
    out['year'] = out['Date'].dt.year
    out['dayofmonth'] = out['Date'].dt.day
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    out['is_month_start'] = out['Date'].dt.is_month_start.astype(int)
    out['is_month_end'] = out['Date'].dt.is_month_end.astype(int)
    holiday_dates = build_vietnam_holiday_dates(range(out['year'].min(), out['year'].max() + 1))
    out['is_holiday'] = out['Date'].isin(holiday_dates).astype(int)
    for col in out.columns.drop('Date'):
        feature_group = 'calendar_flag' if col.startswith('is_') else 'calendar'
        logic = 'So sanh Date voi list ngay le Viet Nam' if col == 'is_holiday' else 'Lay truc tiep tu Date'
        add_feature_record(col, 'Date', feature_group, logic, False, 'low', 'Mua sam co the thay doi theo thu, thang, cuoi tuan va ngay le')
    return out

def make_time_features(daily_df, date_col, feature_specs, prefix, source_table, feature_group, business_reason, leakage_risk='low'):
    """Create only requested lag/rolling features.

    feature_specs example:
    {'col': 'sessions', 'lags': [1], 'roll_mean_windows': [7, 28]}
    No automatic rolling_sum is created because sums are often redundant or wrong for rates/ratios.
    """
    value_cols = [spec['col'] for spec in feature_specs]
    base = daily_df[[date_col] + value_cols].copy()
    base[date_col] = pd.to_datetime(base[date_col], errors='coerce')
    base = base.sort_values(date_col).drop_duplicates(date_col).reset_index(drop=True)
    out = base[[date_col]].rename(columns={date_col: 'Date'})

    for spec in feature_specs:
        col = spec['col']
        s = pd.to_numeric(base[col], errors='coerce')
        for lag in spec.get('lags', []):
            name = f'{prefix}_{col}_lag_{lag}'
            out[name] = s.shift(lag).values
            add_feature_record(name, source_table, feature_group, f'{col} shift {lag} ngay', True, leakage_risk, business_reason)
        shifted = s.shift(1)
        for window in spec.get('roll_mean_windows', []):
            min_periods = spec.get('min_periods', max(2, window // 3))
            name = f'{prefix}_{col}_roll_mean_{window}'
            out[name] = shifted.rolling(window, min_periods=min_periods).mean().values
            add_feature_record(name, source_table, feature_group, f'Mean {col} cua {window} ngay truoc', True, leakage_risk, business_reason)
    return out

def daily_coverage_rate(daily_df, date_col, modeling_dates):
    source_dates = pd.to_datetime(daily_df[date_col], errors='coerce').dropna().dt.normalize().drop_duplicates()
    target_dates = pd.Series(pd.to_datetime(modeling_dates).dt.normalize().unique())
    if len(target_dates) == 0:
        return 0, 0
    # Coverage phai tinh theo phan tram ngay target co du lieu source, khong phai nguoc lai.
    return target_dates.isin(set(source_dates)).mean(), len(source_dates)

def merge_features(modeling_df, feature_df, source_name):
    before_rows = len(modeling_df)
    before_cols = modeling_df.shape[1]
    out = modeling_df.merge(feature_df, on='Date', how='left', validate='one_to_one')
    if len(out) != before_rows:
        raise ValueError(f'Join {source_name} lam thay doi so dong: {before_rows} -> {len(out)}')
    print(f'Merged {source_name}: +{out.shape[1] - before_cols} cols')
    return out

def review_feature_section(section_name, feature_groups, target_columns=('Revenue', 'COGS'), signal_threshold=0.05, strong_threshold=0.10, max_missing=0.25, max_plots=4):
    """Review feature candidates immediately after each 4.x source block."""
    if not feature_records:
        print(f'{section_name}: chua co feature record de review')
        return pd.DataFrame()

    records = pd.DataFrame(feature_records).drop_duplicates('feature_name', keep='last')
    feature_names = records.loc[records['feature_group'].isin(feature_groups), 'feature_name'].tolist()
    feature_names = [c for c in feature_names if c in modeling.columns]
    if not feature_names:
        print(f'{section_name}: khong co feature nao trong modeling de review')
        return pd.DataFrame()

    rows = []
    for feature in feature_names:
        if not pd.api.types.is_numeric_dtype(modeling[feature]):
            continue
        missing_rate = modeling[feature].isna().mean()
        n_unique = modeling[feature].nunique(dropna=True)
        for target in target_columns:
            valid_pair = modeling[[feature, target]].dropna()
            corr = valid_pair[feature].corr(valid_pair[target]) if len(valid_pair) >= 30 and n_unique > 1 else np.nan
            abs_corr = abs(corr) if pd.notna(corr) else np.nan
            if missing_rate > max_missing:
                decision = 'not_selected_now'
                reason = 'missing cao trong section review'
            elif n_unique <= 1:
                decision = 'not_selected_now'
                reason = 'feature gan nhu constant'
            elif pd.notna(abs_corr) and abs_corr >= strong_threshold:
                decision = 'candidate_keep'
                reason = 'signal ro voi target, can xem plot de tranh outlier/leakage'
            elif pd.notna(abs_corr) and abs_corr >= signal_threshold:
                decision = 'candidate_review'
                reason = 'signal vua/yeu nhung co the giu neu nghiep vu ung ho'
            else:
                decision = 'not_selected_now'
                reason = 'chua thay signal du manh voi target trong review tai cho'
            rows.append({
                'section': section_name,
                'feature_name': feature,
                'target': target,
                'missing_rate': missing_rate,
                'n_unique': n_unique,
                'corr_with_target': corr,
                'abs_corr_with_target': abs_corr,
                'section_decision': decision,
                'section_reason': reason,
                'manual_plot_conclusion': '',
            })

    review_df = pd.DataFrame(rows)
    if review_df.empty:
        print(f'{section_name}: khong co numeric feature de review')
        return review_df

    feature_decision = (
        review_df.sort_values('abs_corr_with_target', ascending=False)
        .groupby(['section', 'feature_name'], as_index=False)
        .agg(
            max_abs_corr=('abs_corr_with_target', 'max'),
            best_target=('target', 'first'),
            missing_rate=('missing_rate', 'first'),
            n_unique=('n_unique', 'first'),
            section_decision=('section_decision', lambda s: 'candidate_keep' if (s == 'candidate_keep').any() else ('candidate_review' if (s == 'candidate_review').any() else 'not_selected_now')),
            section_reason=('section_reason', 'first'),
        )
    )
    n_candidate = int(feature_decision['section_decision'].isin(['candidate_keep', 'candidate_review']).sum())
    if n_candidate == len(feature_decision):
        section_conclusion = 'chon_tat_ca_feature_trong_muc'
    elif n_candidate > 0:
        section_conclusion = 'chon_mot_phan_feature_trong_muc'
    else:
        section_conclusion = 'chua_chon_feature_nao_trong_muc'
    feature_decision['section_conclusion'] = section_conclusion

    display(feature_decision.sort_values('max_abs_corr', ascending=False))
    display(pd.DataFrame([{
        'section': section_name,
        'n_features_reviewed': len(feature_decision),
        'n_candidate_after_section_review': n_candidate,
        'section_conclusion': section_conclusion,
        'note': 'Sua manual_plot_conclusion sau khi xem bieu do neu pattern khong hop ly nghiep vu',
    }]))

    section_feature_reviews[:] = [r for r in section_feature_reviews if r.get('section') != section_name]
    section_feature_reviews.extend(review_df.to_dict('records'))

    plot_candidates = review_df.sort_values('abs_corr_with_target', ascending=False).drop_duplicates('feature_name').head(max_plots)
    for row in plot_candidates.itertuples(index=False):
        feature = row.feature_name
        target = row.target
        plot_df = modeling[['Date', feature, target]].dropna().copy()
        if len(plot_df) < 30 or plot_df[feature].nunique() <= 1:
            print(f'Skip plot {feature} -> {target}: khong du data hoac feature gan nhu constant')
            continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        sns.scatterplot(data=plot_df, x=feature, y=target, alpha=0.55, s=24, ax=axes[0])
        axes[0].set_title(f'{section_name}: {feature} vs {target}')
        axes[0].tick_params(axis='x', rotation=30)

        if plot_df[feature].nunique() <= 10:
            sns.boxplot(data=plot_df, x=feature, y=target, ax=axes[1])
            axes[1].set_title(f'{target} distribution by {feature}')
            axes[1].tick_params(axis='x', rotation=30)
        else:
            n_bins = min(6, plot_df[feature].nunique())
            plot_df['feature_bin'] = pd.qcut(plot_df[feature], q=n_bins, duplicates='drop')
            bin_summary = plot_df.groupby('feature_bin', observed=True).agg(
                feature_median=(feature, 'median'),
                target_mean=(target, 'mean'),
                n_rows=(target, 'size'),
            ).reset_index()
            sns.lineplot(data=bin_summary, x='feature_median', y='target_mean', marker='o', ax=axes[1])
            axes[1].set_title(f'Mean {target} by {feature} quantile')
            axes[1].set_xlabel(f'{feature} median in bin')
            axes[1].set_ylabel(f'Mean {target}')
        plt.tight_layout()
        plt.show()

    return feature_decision


## 4. Feature Engineering By Data Source

### 4.1. Historical Target Features

**Mục tiêu:** Tạo lag/rolling từ `Revenue` và `COGS` của quá khứ; bước này giúp model học xu hướng và tính mùa vụ mà không nhìn thấy target của ngày cần dự đoán.


In [ ]:
target_features = make_time_features(
    sales[['Date', 'Revenue', 'COGS']],
    date_col='Date',
    feature_specs=[
        {'col': 'Revenue', 'lags': [1, 7, 28], 'roll_mean_windows': [7, 28]},
        {'col': 'COGS', 'lags': [1, 7, 28], 'roll_mean_windows': [7, 28]},
    ],
    prefix='target',
    source_table='sales',
    feature_group='target_history',
    business_reason='Revenue/COGS qua khu la baseline tin cay cho du bao ngay tiep theo'
)
modeling = merge_features(modeling, target_features, 'target_history')

display(modeling[['Date', 'Revenue', 'COGS'] + [c for c in modeling.columns if c.startswith('target_')]].head(12))
review_feature_section('4.1 Historical Target Features', ['target_history'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.1:** chọn tất cả candidate ở bước review tại chỗ.
- Các feature lịch sử target có tín hiệu mạnh nhất: `target_Revenue_lag_1` corr khoảng `0.865` với `Revenue`, `target_COGS_lag_1` corr khoảng `0.857` với `COGS`; rolling 7/28 ngày cũng có signal rõ.
- Biểu đồ scatter và binned mean cho thấy quan hệ tăng khá đều: ngày trước/rolling quá khứ càng cao thì target hiện tại thường cao hơn.
- Lưu ý: nhóm này rất mạnh nhưng cũng dễ áp đảo model, nên bước sau vẫn dùng `group_cap` để chỉ giữ nhóm đại diện tốt nhất thay vì đưa hết vô model cuối.

### 4.2. Calendar Features

**Mục tiêu:** Tạo feature từ lịch như day of week, month, weekend, đầu/cuối tháng và ngày lễ Việt Nam; bước này giúp model nhận diện chu kỳ mua sắm và các ngày đặc biệt có thể ảnh hưởng doanh thu.


In [ ]:
calendar_features = add_calendar_features(modeling)
modeling = merge_features(modeling, calendar_features, 'calendar')

display(calendar_features.head())
review_feature_section('4.2 Calendar Features', ['calendar', 'calendar_flag'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.2:** chọn một phần feature.
- Giữ làm candidate: `year`, `dayofmonth`, `quarter`, `month`, `is_month_end`, `is_month_start`, `is_weekend`, `dayofweek`.
- Không chọn tại bước review: `is_holiday`, vì corr chỉ khoảng `0.034`, chưa thấy ảnh hưởng đủ rõ với `Revenue/COGS` trong dữ liệu hiện tại.
- Biểu đồ `year` cho thấy target thay đổi mạnh theo giai đoạn, nhưng đây có thể là trend/period effect chứ không hẳn quan hệ nhân quả. Vì vậy `year` cần qua validation/rule phía sau, không tự động xem là feature tốt cuối cùng.

### 4.3. Web Traffic Features

**Mục tiêu:** Aggregate traffic theo ngày và tạo lag/rolling có chọn lọc cho sessions, visitors, page views và engagement; bước này giúp model dùng tín hiệu nhu cầu online trước đó thay vì sinh lag cho mọi cột một cách máy móc.


In [ ]:
if 'web_traffic' in tables:
    traffic = tables['web_traffic'].copy()
    traffic['date'] = pd.to_datetime(traffic['date'], errors='coerce')
    traffic_rules = {
        'unique_visitors_gt_sessions': int((traffic['unique_visitors'] > traffic['sessions']).sum()),
        'page_views_lt_sessions': int((traffic['page_views'] < traffic['sessions']).sum()),
        'bounce_rate_outside_0_1': int((~traffic['bounce_rate'].between(0, 1)).sum()),
    }
    print('Traffic rule check:', traffic_rules)

    traffic_cols = ['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']
    traffic_daily = traffic.groupby('date', as_index=False)[traffic_cols].mean()
    traffic_specs = [
        {'col': 'sessions', 'lags': [1], 'roll_mean_windows': [7]},
        {'col': 'unique_visitors', 'lags': [1], 'roll_mean_windows': [7]},
        {'col': 'page_views', 'lags': [1], 'roll_mean_windows': [7]},
        {'col': 'bounce_rate', 'lags': [1], 'roll_mean_windows': [7]},
        {'col': 'avg_session_duration_sec', 'lags': [1], 'roll_mean_windows': [7]},
    ]
    traffic_features = make_time_features(
        traffic_daily,
        date_col='date',
        feature_specs=traffic_specs,
        prefix='traffic',
        source_table='web_traffic',
        feature_group='web_traffic',
        business_reason='Traffic qua khu co the bao hieu nhu cau mua sam'
    )
    modeling = merge_features(modeling, traffic_features, 'web_traffic')
else:
    print('Khong co web_traffic')

review_feature_section('4.3 Web Traffic Features', ['web_traffic'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.3:** chọn một phần feature.
- Giữ làm candidate: `traffic_sessions_lag_1`, `traffic_sessions_roll_mean_7`, `traffic_unique_visitors_lag_1`, `traffic_unique_visitors_roll_mean_7`, `traffic_page_views_lag_1`, `traffic_page_views_roll_mean_7`.
- Không chọn tại bước review: `traffic_bounce_rate_*` và `traffic_avg_session_duration_sec_*`, vì corr rất thấp, khoảng `0.018-0.026`.
- Plot traffic cho thấy tín hiệu có nhưng không tuyến tính hoàn toàn: COGS/Revenue tăng ở vùng traffic trung bình-cao rồi không tăng đều ở bin cao nhất. Vì vậy chỉ giữ các biến volume traffic, không giữ engagement/rate yếu.

### 4.4. Order Behavior Features

**Mục tiêu:** Aggregate hành vi order theo ngày và tạo feature cho order count/unique customers; bước này giúp model bắt được sức mua gần đây mà không đưa trực tiếp doanh thu cùng ngày vào feature.


In [ ]:
if 'orders' in tables:
    orders = tables['orders'].copy()
    orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
    orders_daily = orders.groupby('order_date').agg(
        order_count=('order_id', 'nunique'),
        order_unique_customers=('customer_id', 'nunique'),
    ).reset_index()

    order_specs = [
        {'col': 'order_count', 'lags': [1, 7], 'roll_mean_windows': [7]},
        {'col': 'order_unique_customers', 'lags': [1, 7], 'roll_mean_windows': [7]},
    ]
    order_features = make_time_features(
        orders_daily,
        date_col='order_date',
        feature_specs=order_specs,
        prefix='orders',
        source_table='orders',
        feature_group='orders',
        business_reason='So order/customer qua khu la demand signal truc tiep nhung khong dung order cung ngay'
    )
    modeling = merge_features(modeling, order_features, 'orders')
else:
    print('Khong co orders')

review_feature_section('4.4 Order Behavior Features', ['orders'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.4:** chọn tất cả candidate ở bước review tại chỗ.
- `orders_order_count_lag_1` và `orders_order_unique_customers_lag_1` có signal rất mạnh, corr khoảng `0.84` với `COGS`.
- Rolling 7 ngày cũng mạnh, corr khoảng `0.64`; lag 7 yếu hơn nhưng vẫn có tín hiệu, corr khoảng `0.43`.
- Biểu đồ cho thấy quan hệ tăng rõ và hợp lý nghiệp vụ: số order/unique customer quá khứ cao thì target ngày hiện tại thường cao hơn. Bước final selection vẫn dùng `group_cap` để tránh giữ quá nhiều feature order tương tự nhau.

### 4.5. Product & Order Item Features

**Mục tiêu:** Tạo feature về demand mix, quantity, promo usage và tỉ trọng category từ order items/products; bước này giúp model hiểu cơ cấu sản phẩm bán ra nhưng tránh dùng các proxy quá sát target như line revenue mặc định.


In [ ]:
if {'orders', 'order_items', 'products'}.issubset(tables):
    products = tables['products'].copy()
    order_items = tables['order_items'].copy()
    item_key = ['order_id', 'product_id']
    if not set(item_key).issubset(order_items.columns):
        print('Bo qua product feature vi order_items thieu order_id/product_id')
        record_skipped_source('order_items + products', 'order_items thieu key order_id/product_id')
    elif products.duplicated(['product_id']).any():
        print('Bo qua product feature vi products.product_id khong unique')
        record_skipped_source('order_items + products', 'products.product_id khong unique')
    else:
        duplicate_item_rows = order_items[order_items.duplicated(item_key, keep=False)].sort_values(item_key)
        order_items_key_check = pd.DataFrame([{
            'table': 'order_items',
            'expected_grain': '1 row = 1 order_id x product_id',
            'key_columns': ', '.join(item_key),
            'n_rows': len(order_items),
            'n_unique_key': order_items.drop_duplicates(item_key).shape[0],
            'duplicate_key_rows': int(order_items.duplicated(item_key).sum()),
            'exact_duplicate_rows': int(order_items.duplicated().sum()),
            'decision': 'aggregate duplicate keys before daily feature' if len(duplicate_item_rows) else 'use as-is',
        }])
        display(order_items_key_check)

        if len(duplicate_item_rows):
            display(duplicate_item_rows.head(30))
            order_items = order_items.groupby(item_key, as_index=False).agg(
                quantity=('quantity', 'sum'),
                unit_price=('unit_price', 'mean'),
                discount_amount=('discount_amount', 'sum'),
                promo_id=('promo_id', 'first'),
                promo_id_2=('promo_id_2', 'first'),
            )

        order_dates = tables['orders'][['order_id', 'order_date']].copy()
        order_dates['order_date'] = pd.to_datetime(order_dates['order_date'], errors='coerce')
        items = order_items.merge(order_dates, on='order_id', how='left', validate='many_to_one')
        items = items.merge(products[['product_id', 'category', 'segment']], on='product_id', how='left', validate='many_to_one')
        items['has_promo'] = items['promo_id'].notna().astype(int)
        items['has_second_promo'] = items['promo_id_2'].notna().astype(int)

        item_daily = items.groupby('order_date').agg(
            item_quantity_sum=('quantity', 'sum'),
            item_product_unique=('product_id', 'nunique'),
            item_discount_mean=('discount_amount', 'mean'),
            item_has_promo_rate=('has_promo', 'mean'),
            item_has_second_promo_rate=('has_second_promo', 'mean'),
        ).reset_index()

        top_categories = items.groupby('category')['quantity'].sum().sort_values(ascending=False).head(4).index.tolist()
        category_qty = items[items['category'].isin(top_categories)].pivot_table(
            index='order_date',
            columns='category',
            values='quantity',
            aggfunc='sum',
            fill_value=0,
        )
        daily_quantity = items.groupby('order_date')['quantity'].sum().replace(0, np.nan)
        category_share = category_qty.div(daily_quantity, axis=0)
        category_share = category_share.add_prefix('item_category_quantity_share_').reset_index()
        item_daily = item_daily.merge(category_share, on='order_date', how='left', validate='one_to_one')

        item_specs = [
            {'col': 'item_quantity_sum', 'lags': [1, 7], 'roll_mean_windows': [7, 28]},
            {'col': 'item_product_unique', 'lags': [1], 'roll_mean_windows': [7]},
            {'col': 'item_discount_mean', 'lags': [1], 'roll_mean_windows': [7]},
            {'col': 'item_has_promo_rate', 'lags': [1], 'roll_mean_windows': [7]},
            {'col': 'item_has_second_promo_rate', 'lags': [1], 'roll_mean_windows': [7]},
        ]
        for col in [c for c in item_daily.columns if c.startswith('item_category_quantity_share_')]:
            item_specs.append({'col': col, 'lags': [1], 'roll_mean_windows': [7]})

        item_features = make_time_features(
            item_daily,
            date_col='order_date',
            feature_specs=item_specs,
            prefix='items',
            source_table='order_items + products',
            feature_group='products_sales_mix',
            business_reason='Quantity, promo va category mix qua khu giup mo ta demand; khong dung line revenue/cogs proxy target'
        )
        modeling = merge_features(modeling, item_features, 'order_items_products')
else:
    print('Khong co du orders/order_items/products')

review_feature_section('4.5 Product & Order Item Features', ['products_sales_mix'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.5:** chọn một phần feature.
- Giữ mạnh nhất: `items_item_product_unique_lag_1`, `items_item_quantity_sum_lag_1`, `items_item_product_unique_roll_mean_7`, `items_item_quantity_sum_roll_mean_7`, `items_item_quantity_sum_roll_mean_28`, `items_item_quantity_sum_lag_7`.
- Giữ thêm để review mix sản phẩm: share `Streetwear`, `Outdoor`, `Casual`, `item_discount_mean`, `item_has_promo_rate` khi signal đạt ngưỡng.
- Không chọn tại bước review: share `GenZ` và `item_has_second_promo_rate_*`, vì corr rất thấp.
- Plot `item_product_unique_lag_1` và `item_quantity_sum_lag_1` tăng rất rõ với target, phù hợp nghiệp vụ demand. Nhưng các biến này gần với order demand, nên heatmap/rule phía sau cần kiểm tra trùng tín hiệu với `orders_*` và `target_*`.

### 4.6. Promotion Features

**Mục tiêu:** Tạo feature promotion active theo ngày từ lịch khuyến mãi có biết trước; bước này giúp model nắm bắt tác động của khuyến mãi mà không cần lag vì đây là thông tin plan trước ngày dự báo.


In [ ]:
if 'promotions' in tables:
    promos = tables['promotions'].copy()
    promos['start_date'] = pd.to_datetime(promos['start_date'], errors='coerce')
    promos['end_date'] = pd.to_datetime(promos['end_date'], errors='coerce')
    promo_feature_rows = []
    for d in modeling['Date']:
        active = promos[(promos['start_date'] <= d) & (promos['end_date'] >= d)]
        promo_feature_rows.append({
            'Date': d,
            'promo_active_count': len(active),
            'promo_discount_mean': active['discount_value'].mean() if len(active) else 0,
            'promo_min_order_value_mean': active['min_order_value'].mean() if len(active) else 0,
            'promo_stackable_count': active['stackable_flag'].sum() if len(active) else 0,
        })
    promo_features = pd.DataFrame(promo_feature_rows)
    for col in promo_features.columns.drop('Date'):
        add_feature_record(col, 'promotions', 'promotions', 'Promotion active theo Date', False, 'low', 'Khuyen mai co the lam thay doi demand va margin')
    modeling = merge_features(modeling, promo_features, 'promotions')
else:
    print('Khong co promotions')

review_feature_section('4.6 Promotion Features', ['promotions'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.6:** chọn một phần feature nhưng cần review kỹ.
- `promo_discount_mean` là candidate rõ nhất, corr khoảng `0.144` với `Revenue`.
- `promo_active_count`, `promo_min_order_value_mean`, `promo_stackable_count` chỉ ở mức candidate review, corr khoảng `0.064-0.095`.
- Plot promotion cho thấy quan hệ dạng nhóm/rời rạc, không tăng tuyến tính theo discount. Một số mức discount có doanh thu cao hơn, nhưng không đủ để kết luận nhân quả mạnh. Vì vậy promotion được giữ thận trọng và phải qua validation.

### 4.7. Inventory Features

**Mục tiêu:** Kiểm tra coverage inventory trước khi dùng làm feature; bước này giúp tránh đưa vào model một source quá thưa ngày, gây missing lớn và làm tín hiệu không ổn định.


In [ ]:
if 'inventory' in tables:
    inv = tables['inventory'].copy()
    inv['snapshot_date'] = pd.to_datetime(inv['snapshot_date'], errors='coerce')
    inv_daily = inv.groupby('snapshot_date').agg(
        inv_stock_on_hand_sum=('stock_on_hand', 'sum'),
        inv_fill_rate_mean=('fill_rate', 'mean'),
        inv_stockout_rate=('stockout_flag', 'mean'),
    ).reset_index()
    inv_coverage, inv_n_dates = daily_coverage_rate(inv_daily, 'snapshot_date', modeling['Date'])
    print(f'Inventory date coverage: {inv_coverage:.2%} ({inv_n_dates} source dates)')

    if inv_coverage < 0.60:
        record_skipped_source('inventory', 'skip vi coverage theo Date qua thap, neu join se tao feature missing nhieu', inv_coverage, inv_n_dates)
        display(pd.DataFrame(skipped_feature_sources).tail(1))
    else:
        inv_specs = [
            {'col': 'inv_stock_on_hand_sum', 'lags': [1], 'roll_mean_windows': [7]},
            {'col': 'inv_fill_rate_mean', 'lags': [1], 'roll_mean_windows': [7]},
            {'col': 'inv_stockout_rate', 'lags': [1], 'roll_mean_windows': [7]},
        ]
        inv_features = make_time_features(
            inv_daily,
            date_col='snapshot_date',
            feature_specs=inv_specs,
            prefix='inventory',
            source_table='inventory',
            feature_group='inventory',
            business_reason='Ton kho va stockout qua khu co the anh huong kha nang ban hang'
        )
        modeling = merge_features(modeling, inv_features, 'inventory')
else:
    print('Khong co inventory')

review_feature_section('4.7 Inventory Features', ['inventory'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.7:** không chọn feature inventory ở lần chạy này.
- Lý do: coverage theo ngày của inventory chỉ khoảng `3.29%` so với modeling date, thấp hơn ngưỡng `60%` trong notebook.
- Nếu join inventory vào daily modeling sẽ tạo missing rất lớn, làm model input không ổn định. Vì vậy source này được ghi vào skipped source thay vì tạo feature.

### 4.8. Fulfillment, Return & Review Features

**Mục tiêu:** Tạo feature lag/rolling cho shipment, return và review; bước này giúp model có tín hiệu hậu mua hàng trong quá khứ, đồng thời tránh leakage vì các event này thường xảy ra sau order.


In [ ]:
if 'shipments' in tables:
    ship = tables['shipments'].copy()
    ship['ship_date'] = pd.to_datetime(ship['ship_date'], errors='coerce')
    ship['delivery_date'] = pd.to_datetime(ship['delivery_date'], errors='coerce')
    ship['delivery_delay_days'] = (ship['delivery_date'] - ship['ship_date']).dt.days
    ship_daily = ship.groupby('ship_date').agg(
        ship_count=('order_id', 'nunique'),
        ship_delivery_delay_mean=('delivery_delay_days', 'mean'),
    ).reset_index()
    ship_features = make_time_features(
        ship_daily,
        date_col='ship_date',
        feature_specs=[
            {'col': 'ship_count', 'lags': [1], 'roll_mean_windows': [7, 28]},
            {'col': 'ship_delivery_delay_mean', 'roll_mean_windows': [28]},
        ],
        prefix='ship',
        source_table='shipments',
        feature_group='operations',
        business_reason='Van hanh giao hang qua khu co the phan anh suc mua va chat luong dich vu',
        leakage_risk='medium'
    )
    modeling = merge_features(modeling, ship_features, 'shipments')

if 'returns' in tables:
    ret = tables['returns'].copy()
    ret['return_date'] = pd.to_datetime(ret['return_date'], errors='coerce')
    ret_daily = ret.groupby('return_date').agg(
        return_count=('return_id', 'nunique'),
        refund_amount_sum=('refund_amount', 'sum'),
    ).reset_index()
    ret_features = make_time_features(
        ret_daily,
        date_col='return_date',
        feature_specs=[
            {'col': 'return_count', 'roll_mean_windows': [28]},
            {'col': 'refund_amount_sum', 'roll_mean_windows': [28]},
        ],
        prefix='return',
        source_table='returns',
        feature_group='post_purchase',
        business_reason='Return/refund qua khu co the phan anh chat luong ban hang',
        leakage_risk='medium'
    )
    modeling = merge_features(modeling, ret_features, 'returns')

if 'reviews' in tables:
    reviews = tables['reviews'].copy()
    reviews['review_date'] = pd.to_datetime(reviews['review_date'], errors='coerce')
    review_daily = reviews.groupby('review_date').agg(
        review_count=('review_id', 'nunique'),
        review_rating_mean=('rating', 'mean'),
    ).reset_index()
    review_features = make_time_features(
        review_daily,
        date_col='review_date',
        feature_specs=[
            {'col': 'review_count', 'roll_mean_windows': [28]},
            {'col': 'review_rating_mean', 'roll_mean_windows': [28]},
        ],
        prefix='review',
        source_table='reviews',
        feature_group='customer_feedback',
        business_reason='Review qua khu co the phan anh trai nghiem khach hang',
        leakage_risk='medium'
    )
    modeling = merge_features(modeling, review_features, 'reviews')

review_feature_section('4.8 Fulfillment, Return & Review Features', ['operations', 'post_purchase', 'customer_feedback'])


#### Kết luận sau khi chạy output

- **Chốt mục 4.8:** chọn một phần feature.
- Giữ candidate: `ship_ship_count_lag_1`, `ship_ship_count_roll_mean_7`, `ship_ship_count_roll_mean_28`, `return_return_count_roll_mean_28`, `return_refund_amount_sum_roll_mean_28`, `review_review_count_roll_mean_28`.
- Không chọn tại bước review: `ship_ship_delivery_delay_mean_roll_mean_28` và `review_review_rating_mean_roll_mean_28`, vì corr dưới ngưỡng `0.05`.
- Plot shipment count có quan hệ tăng rõ với target, nhưng nhóm fulfillment/return/review có `leakage_risk='medium'` vì là tín hiệu hậu mua hàng. Chỉ dùng lag/rolling quá khứ và cần validation kỹ.

## 5. Feature Consolidation

### 5.1. Daily Feature Join

**Mục tiêu:** Kiểm tra bảng modeling sau khi join tất cả feature theo `Date`; bước này giúp xác nhận số dòng vẫn bằng với target daily và không bị nhân dòng sau join.


In [ ]:
modeling_table_check = pd.DataFrame([{
    'n_rows': len(modeling),
    'n_unique_date': modeling['Date'].nunique(),
    'duplicate_date_rows': int(modeling['Date'].duplicated().sum()),
    'n_columns': modeling.shape[1],
    'date_min': modeling['Date'].min(),
    'date_max': modeling['Date'].max(),
}])
display(modeling_table_check)

if modeling['Date'].duplicated().any():
    raise ValueError('Modeling table bi duplicate Date sau khi join feature')


### 5.2. Post-Join Missing Check

**Mục tiêu:** Review missing sau join theo từng feature; bước này giúp biết feature nào mất coverage, cần fill có chú ý, loại bỏ, hoặc đánh dấu để review.


In [ ]:
feature_cols_preview = [c for c in modeling.columns if c not in ['Date', 'Revenue', 'COGS']]
missing_after_join = pd.DataFrame({
    'feature_name': feature_cols_preview,
    'missing_rate': [modeling[c].isna().mean() for c in feature_cols_preview],
    'n_missing': [int(modeling[c].isna().sum()) for c in feature_cols_preview],
}).sort_values('missing_rate', ascending=False)

display(missing_after_join.head(30))


### 5.3. Section-Level Feature Decision Summary

**Mục tiêu:** Tổng hợp kết luận sơ bộ từ từng mục `4.x` trước khi đi qua heatmap, quality rule và validation-based selection. Bước này giúp feature selection không bị gom mù ở cuối notebook: mỗi nhóm nguồn đã có review riêng và danh sách candidate riêng.


In [ ]:
target_cols = ['Revenue', 'COGS']
feature_cols = [c for c in modeling.columns if c not in ['Date'] + target_cols]

if section_feature_reviews:
    section_review_df = pd.DataFrame(section_feature_reviews)
    section_feature_level_decision = (
        section_review_df.sort_values('abs_corr_with_target', ascending=False)
        .groupby(['section', 'feature_name'], as_index=False)
        .agg(
            max_abs_corr=('abs_corr_with_target', 'max'),
            best_target=('target', 'first'),
            missing_rate_section_review=('missing_rate', 'first'),
            section_decision=('section_decision', lambda s: 'candidate_keep' if (s == 'candidate_keep').any() else ('candidate_review' if (s == 'candidate_review').any() else 'not_selected_now')),
            section_reason=('section_reason', 'first'),
            manual_plot_conclusion=('manual_plot_conclusion', 'first'),
        )
    )
    section_preselected_feature_names = section_feature_level_decision.loc[
        section_feature_level_decision['section_decision'].isin(['candidate_keep', 'candidate_review']),
        'feature_name'
    ].drop_duplicates().tolist()

    section_summary = section_feature_level_decision.groupby('section').agg(
        n_features_reviewed=('feature_name', 'nunique'),
        n_candidate=('section_decision', lambda s: int(s.isin(['candidate_keep', 'candidate_review']).sum())),
    ).reset_index()
    section_summary['section_conclusion'] = np.select(
        [
            section_summary['n_candidate'] == section_summary['n_features_reviewed'],
            section_summary['n_candidate'] > 0,
        ],
        ['chon_tat_ca_feature_trong_muc', 'chon_mot_phan_feature_trong_muc'],
        default='chua_chon_feature_nao_trong_muc'
    )

    display(section_summary)
    display(section_feature_level_decision.sort_values(['section', 'section_decision', 'max_abs_corr'], ascending=[True, True, False]))
else:
    section_review_df = pd.DataFrame()
    section_feature_level_decision = pd.DataFrame(columns=['section', 'feature_name', 'max_abs_corr', 'best_target', 'section_decision', 'section_reason', 'manual_plot_conclusion'])
    section_preselected_feature_names = []
    print('Chua co section review nao tu buoc 4.x')

print('Section-preselected features:', len(section_preselected_feature_names))


#### Kết luận tổng hợp sau section review

- Sau khi chạy review từng mục `4.x`, danh sách candidate sơ bộ được lấy từ `section_decision in ['candidate_keep', 'candidate_review']`.
- Inventory không được đưa tiếp vì coverage quá thấp.
- Heatmap tổng thể cho thấy các nhóm `target_history`, `orders`, `products_sales_mix`, `shipments/returns` tương quan rất cao với nhau. Điều này hợp lý vì đều là demand signal, nhưng cũng có nguy cơ trùng thông tin.
- Vì vậy các bước sau không chỉ dựa vào corr tại chỗ: vẫn cần `quality_status`, `leakage_risk`, `group_cap` và train/validation stability để lọc bớt feature trùng hoặc rủi ro.

## 6. Feature Quality Review

### 6.1. Missing, Constant & Invalid Feature Check

**Mục tiêu:** Tạo feature catalog với missing rate, constant flag và status; bước này giúp lọc bớt feature kém chất lượng trước khi đánh giá tín hiệu với target.


In [ ]:
target_cols = ['Revenue', 'COGS']
feature_cols = [c for c in modeling.columns if c not in ['Date'] + target_cols]

quality_rows = []
for col in feature_cols:
    s = modeling[col]
    quality_rows.append({
        'feature_name': col,
        'missing_rate': float(s.isna().mean()),
        'n_unique': int(s.nunique(dropna=True)),
        'dtype': str(s.dtype),
        'is_constant': s.nunique(dropna=True) <= 1,
    })
feature_quality = pd.DataFrame(quality_rows)

catalog = pd.DataFrame(feature_records).drop_duplicates('feature_name', keep='last')
catalog = catalog.merge(feature_quality, on='feature_name', how='left')
catalog = catalog.merge(
    section_feature_level_decision[[
        'feature_name', 'section', 'max_abs_corr', 'best_target',
        'section_decision', 'section_reason', 'manual_plot_conclusion'
    ]],
    on='feature_name',
    how='left'
)
catalog['section_decision'] = catalog['section_decision'].fillna('not_reviewed_in_section')
catalog['quality_status'] = np.where(
    catalog['is_constant'], 'drop_constant',
    np.where(catalog['missing_rate'] > 0.25, 'review_high_missing', 'pass')
)

skipped_feature_sources_df = pd.DataFrame(skipped_feature_sources)
print('Feature count by group:')
display(catalog.groupby('feature_group').size().sort_values(ascending=False).to_frame('n_features'))
if len(skipped_feature_sources_df):
    print('Skipped sources:')
    display(skipped_feature_sources_df)

display(catalog.sort_values(['quality_status', 'missing_rate'], ascending=[True, False]).head(80))


### 6.2. Leakage Risk Review

**Mục tiêu:** Review leakage risk theo feature group và liệt kê feature rủi ro cao nếu có; bước này giúp tránh feature làm validation đẹp giả nhưng không dùng được khi dự báo thật.


In [ ]:
leakage_review = catalog.groupby(['feature_group', 'leakage_risk']).size().reset_index(name='n_features')
display(leakage_review)

high_risk_features = catalog[catalog['leakage_risk'].eq('high')]
display(high_risk_features[['feature_name', 'feature_group', 'leakage_risk']])


### 6.3. Feature EDA

**Mục tiêu:** Xem phân bố và tương quan cơ bản của feature sau khi tạo; bước này giúp phát hiện feature lệch, quá nhiều zero/missing, hoặc có tín hiệu quá bất thường cần review lại logic.


In [ ]:
# EDA nhanh sau khi tao feature: xem missing cao va correlation don gian de review, chua train model.
eda_numeric_cols = modeling[[c for c in modeling.columns if c not in ['Date']]].select_dtypes(include='number').columns.tolist()
eda_summary = modeling[eda_numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

display(eda_summary.head(40))

# Heatmap chi dung sau khi da tong hop tat ca feature vao modeling.
# Khong ve heatmap rieng tung source vi luc do chua thay tuong quan tong the va duplicate signal giua cac nhom.
numeric_feature_cols = [c for c in feature_cols if c in modeling.select_dtypes(include='number').columns]
global_corr_rows = []
for col in numeric_feature_cols:
    for target in target_cols:
        valid_pair = modeling[[col, target]].dropna()
        corr_value = valid_pair[col].corr(valid_pair[target]) if len(valid_pair) >= 30 and valid_pair[col].nunique() > 1 else np.nan
        global_corr_rows.append({
            'feature_name': col,
            'target': target,
            'corr_with_target': corr_value,
            'abs_corr_with_target': abs(corr_value) if pd.notna(corr_value) else np.nan,
        })

global_feature_corr = pd.DataFrame(global_corr_rows)
display(global_feature_corr.sort_values('abs_corr_with_target', ascending=False).head(40))

top_corr_features = (
    global_feature_corr.groupby('feature_name')['abs_corr_with_target']
    .max()
    .sort_values(ascending=False)
    .head(25)
    .index.tolist()
)
heatmap_cols = target_cols + top_corr_features
if len(top_corr_features):
    plt.figure(figsize=(14, max(8, 0.35 * len(heatmap_cols))))
    sns.heatmap(modeling[heatmap_cols].corr(), cmap='vlag', center=0, linewidths=0.2)
    plt.title('Global correlation heatmap after all feature groups are joined')
    plt.tight_layout()
    plt.show()


## 7. Time-Based Train/Validation Split

### 7.1. Train/Validation Period Definition

**Mục tiêu:** Chia train/validation theo thời gian, dùng giai đoạn cuối làm validation; bước này giúp đánh giá gần với bài toán forecast thực tế và tránh leakage từ random split.


In [ ]:
validation_days = 365
last_date = modeling['Date'].max()
validation_start = last_date - pd.Timedelta(days=validation_days - 1)
train_mask = modeling['Date'] < validation_start
valid_mask = modeling['Date'] >= validation_start

split_summary = pd.DataFrame([
    {'split': 'train', 'start_date': modeling.loc[train_mask, 'Date'].min(), 'end_date': modeling.loc[train_mask, 'Date'].max(), 'n_rows': int(train_mask.sum())},
    {'split': 'validation', 'start_date': modeling.loc[valid_mask, 'Date'].min(), 'end_date': modeling.loc[valid_mask, 'Date'].max(), 'n_rows': int(valid_mask.sum())},
])
display(split_summary)

if train_mask.sum() == 0 or valid_mask.sum() == 0:
    raise ValueError('Train/validation split khong hop le')


## 8. Feature Selection

### 8.1. Rule-Based Feature Filtering

**Mục tiêu:** Lọc feature theo quality, missing, leakage và vai trò nghiệp vụ trước khi xem validation signal; bước này giúp giảm feature thừa và tránh để model bị ảnh hưởng bởi biến kém tin cậy.


In [ ]:
rule_based_filter_summary = catalog.groupby(['quality_status', 'leakage_risk']).size().reset_index(name='n_features')
display(rule_based_filter_summary)

rule_filter_candidates = catalog[[
    'feature_name', 'feature_group', 'section', 'section_decision',
    'max_abs_corr', 'best_target', 'missing_rate', 'quality_status', 'leakage_risk'
]].sort_values(['quality_status', 'missing_rate'], ascending=[True, False])
display(rule_filter_candidates.head(40))


### 8.2. Validation-Based Feature Review

**Mục tiêu:** Đánh giá tín hiệu feature trên train/validation và giới hạn số feature theo group; bước này giúp chọn feature có khả năng tổng quát hơn thay vì chỉ mạnh trên một giai đoạn.


In [ ]:
corr_rows = []
numeric_features = modeling[feature_cols].select_dtypes(include='number').columns.tolist()
for col in numeric_features:
    for target in target_cols:
        train_valid = modeling.loc[train_mask, [col, target]].dropna()
        hold_valid = modeling.loc[valid_mask, [col, target]].dropna()
        train_corr = train_valid[col].corr(train_valid[target]) if len(train_valid) >= 30 and train_valid[col].nunique() > 1 else np.nan
        valid_corr = hold_valid[col].corr(hold_valid[target]) if len(hold_valid) >= 30 and hold_valid[col].nunique() > 1 else np.nan
        same_direction = pd.notna(train_corr) and pd.notna(valid_corr) and np.sign(train_corr) == np.sign(valid_corr)
        corr_rows.append({
            'feature_name': col,
            'target': target,
            'train_corr_with_target': train_corr,
            'valid_corr_with_target': valid_corr,
            'abs_train_corr': abs(train_corr) if pd.notna(train_corr) else np.nan,
            'abs_valid_corr': abs(valid_corr) if pd.notna(valid_corr) else np.nan,
            'same_direction_train_valid': same_direction,
        })
corr_df = pd.DataFrame(corr_rows)

selection_base = catalog.merge(corr_df, on='feature_name', how='left')
selection_base['passes_section_review'] = selection_base['section_decision'].isin(['candidate_keep', 'candidate_review'])
selection_base['passes_quality_filter'] = (
    (selection_base['passes_section_review'])
    & (selection_base['quality_status'] == 'pass')
    & (selection_base['leakage_risk'] != 'high')
    & (selection_base['missing_rate'] <= 0.25)
)
selection_base['has_stable_signal'] = (
    (selection_base['abs_train_corr'].fillna(0) >= 0.05)
    & (selection_base['abs_valid_corr'].fillna(0) >= 0.05)
    & (selection_base['same_direction_train_valid'])
)
selection_base['preselected'] = selection_base['passes_quality_filter'] & selection_base['has_stable_signal']

# Gioi han so feature moi nhom/target de tranh mot nhom feature ap dao model.
group_caps = {
    'target_history': 6,
    'calendar': 4,
    'calendar_flag': 4,
    'web_traffic': 6,
    'orders': 4,
    'products_sales_mix': 8,
    'promotions': 4,
    'operations': 3,
    'post_purchase': 3,
    'customer_feedback': 3,
    'inventory': 3,
}
selection_base['group_cap'] = selection_base['feature_group'].map(group_caps).fillna(3).astype(int)
selection_base['rank_in_target_group'] = selection_base.groupby(['target', 'feature_group'])['abs_valid_corr'].rank(method='first', ascending=False)
selection_base['selected'] = selection_base['preselected'] & (selection_base['rank_in_target_group'] <= selection_base['group_cap'])
selection_base['selected_for_target'] = np.where(selection_base['selected'], selection_base['target'], '')
selection_base['selection_reason'] = np.where(
    selection_base['selected'],
    'pass section review + quality + stable train/validation signal + group cap',
    np.where(~selection_base['passes_section_review'], 'not selected in 4.x section review',
             np.where(~selection_base['passes_quality_filter'], 'fail quality/leakage filter',
             np.where(~selection_base['has_stable_signal'], 'weak or unstable validation signal', 'not in top group cap'))
    )
)
selection_base['validation_result'] = np.where(selection_base['same_direction_train_valid'], 'same_direction', 'not_stable_or_not_enough_signal')
selection_base['final_decision'] = np.where(selection_base['selected'], 'candidate_selected', 'not_selected_now')
selection_base['review_note'] = ''

selected_features_summary = selection_base[[
    'feature_name', 'selected_for_target', 'feature_group', 'source_table', 'business_reason',
    'section', 'section_decision', 'section_reason', 'max_abs_corr', 'best_target',
    'missing_rate', 'quality_status', 'leakage_risk', 'selection_reason',
    'train_corr_with_target', 'valid_corr_with_target', 'rank_in_target_group',
    'validation_result', 'final_decision', 'review_note'
]].sort_values(['final_decision', 'selected_for_target', 'feature_group', 'rank_in_target_group'], ascending=[True, True, True, True])

display(selected_features_summary.head(120))


### 8.3. Selected Feature Visual Audit

**Mục tiêu:** Vẽ biểu đồ cho từng feature đã được chọn ở `8.2` để kiểm tra quan hệ với target có hợp lý không, có bị outlier kéo không, và có khác biệt giữa train/validation không. Đây là bước review trực quan sau selection, không thay thế validation metric.


In [ ]:
selected_visual_audit = selected_features_summary[
    (selected_features_summary['final_decision'] == 'candidate_selected')
    & (selected_features_summary['selected_for_target'] != '')
].copy()

selected_visual_audit['abs_valid_corr'] = selected_visual_audit['valid_corr_with_target'].abs()
selected_visual_audit['preliminary_visual_conclusion'] = np.select(
    [
        selected_visual_audit['abs_valid_corr'] >= 0.20,
        selected_visual_audit['abs_valid_corr'] >= 0.10,
        selected_visual_audit['abs_valid_corr'] >= 0.05,
    ],
    [
        'candidate_keep: signal kha ro, can xem plot de loai tru outlier/leakage',
        'candidate_keep_with_review: signal vua, can xem pattern co hop ly nghiep vu khong',
        'weak_candidate: dat nguong toi thieu, chi giu neu plot va nghiep vu ung ho',
    ],
    default='review_or_drop: signal yeu tren validation'
)
selected_visual_audit['manual_visual_note'] = ''

display(selected_visual_audit[[
    'feature_name', 'selected_for_target', 'feature_group',
    'train_corr_with_target', 'valid_corr_with_target',
    'preliminary_visual_conclusion', 'manual_visual_note'
]])

for row in selected_visual_audit.itertuples(index=False):
    feature = row.feature_name
    target = row.selected_for_target
    plot_df = modeling[['Date', feature, target]].dropna().copy()
    if len(plot_df) < 30 or plot_df[feature].nunique() <= 1:
        print(f'Skip plot {feature} -> {target}: khong du data hoac feature gan nhu constant')
        continue

    plot_df['split'] = np.where(plot_df['Date'] >= validation_start, 'validation', 'train')
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.scatterplot(
        data=plot_df,
        x=feature,
        y=target,
        hue='split',
        alpha=0.55,
        s=24,
        ax=axes[0],
    )
    axes[0].set_title(f'{feature} vs {target}')
    axes[0].tick_params(axis='x', rotation=30)

    if plot_df[feature].nunique() <= 10:
        sns.boxplot(data=plot_df, x=feature, y=target, ax=axes[1])
        axes[1].set_title(f'{target} distribution by {feature}')
        axes[1].tick_params(axis='x', rotation=30)
    else:
        n_bins = min(6, plot_df[feature].nunique())
        plot_df['feature_bin'] = pd.qcut(plot_df[feature], q=n_bins, duplicates='drop')
        bin_summary = plot_df.groupby('feature_bin', observed=True).agg(
            feature_median=(feature, 'median'),
            target_mean=(target, 'mean'),
            n_rows=(target, 'size'),
        ).reset_index()
        sns.lineplot(data=bin_summary, x='feature_median', y='target_mean', marker='o', ax=axes[1])
        axes[1].set_title(f'Mean {target} by {feature} quantile')
        axes[1].set_xlabel(f'{feature} median in bin')
        axes[1].set_ylabel(f'Mean {target}')
        for point in bin_summary.itertuples(index=False):
            axes[1].annotate(f'n={int(point.n_rows)}', (point.feature_median, point.target_mean), fontsize=8)

    plt.tight_layout()
    plt.show()


## 9. Required Outputs

### 9.1. Modeling Table

**Mục tiêu:** Lưu bảng modeling daily đã có target và feature; bước này tạo input chính cho notebook train model tiếp theo.


In [ ]:
modeling_table_path = OUTPUT_DIR / 'modeling_table_daily.csv'
modeling.to_csv(modeling_table_path, index=False)
print('Saved modeling table:', modeling_table_path)
print('Modeling shape:', modeling.shape)


### 9.2. Feature Catalog

**Mục tiêu:** Lưu catalog giải thích feature, group, quality và leakage risk; bước này giúp review feature có hệ thống thay vì chỉ nhìn danh sách cột.


In [ ]:
feature_catalog_path = OUTPUT_DIR / 'feature_catalog.csv'
catalog.to_csv(feature_catalog_path, index=False)
print('Saved feature catalog:', feature_catalog_path)
print('Feature catalog rows:', len(catalog))


### 9.3. Selected Feature Summary

**Mục tiêu:** Lưu bảng feature được chọn/không chọn kèm lý do và chỉ số review; bước này giúp minh bạch quyết định feature selection và dễ audit lại.


In [ ]:
selected_summary_path = OUTPUT_DIR / 'selected_features_summary.csv'
selected_features_summary.to_csv(selected_summary_path, index=False)
print('Saved selected feature summary:', selected_summary_path)
print('Candidate selected rows:', int((selected_features_summary['final_decision'] == 'candidate_selected').sum()))


### 9.4. Train/Validation Split Summary

**Mục tiêu:** Lưu thông tin split và source bị skip; bước này giúp người train model biết dữ liệu được chia ra sao và feature source nào đã bị loại vì không đủ tin cậy.


In [ ]:
split_summary_path = OUTPUT_DIR / 'train_validation_split_summary.csv'
skipped_sources_path = OUTPUT_DIR / 'skipped_feature_sources.csv'
split_summary.to_csv(split_summary_path, index=False)
pd.DataFrame(skipped_feature_sources).to_csv(skipped_sources_path, index=False)
print('Saved train/validation split summary:', split_summary_path)
print('Saved skipped feature sources:', skipped_sources_path)


## 10. Completion Criteria

**Mục tiêu:** Kiểm tra tất cả output bắt buộc đã tồn tại và có đủ dòng/cột; bước này giúp xác nhận notebook feature engineering đã hoàn thành trước khi chuyển sang modeling.


In [ ]:
required_output_paths = [
    OUTPUT_DIR / 'modeling_table_daily.csv',
    OUTPUT_DIR / 'feature_catalog.csv',
    OUTPUT_DIR / 'selected_features_summary.csv',
    OUTPUT_DIR / 'train_validation_split_summary.csv',
    OUTPUT_DIR / 'skipped_feature_sources.csv',
]
completion_check = pd.DataFrame({
    'output_path': [str(path) for path in required_output_paths],
    'exists': [path.exists() for path in required_output_paths],
})
display(completion_check)

if not completion_check['exists'].all():
    raise ValueError('Some required outputs are missing')
